In [14]:
import ee
import sys
import random

# Initialize Earth Engine
EE_PROJECT = 'arctic-biomass-mapping'
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

# ====================
# 1. SET UP ==========
# ====================

# 1.0 PARAMETERS ==========

fire_size_thresh_acre = 10
version = 'v20260512'
fire_start_yr = 1940
fire_end_yr = 2025
scale = 30

# 1.1 ROI DATA ==========

roi = ee.FeatureCollection("projects/fisl-tundra-fire/assets/rois/above_arctic_simple")
roi_mask = ee.Image('projects/foreststructure/ABoVE/CCDC/CCDC_ABoVE_L4578SR_1984_2020_J091_273_v20201203_mappedArea_na_beringia_westCan')

# 1.2 FIRE DATA ==========

fires_historic = ee.ImageCollection("projects/fisl-tundra-fire/assets/fires_raster/akca_fires_30m_3338_arctic")
fires_modern = ee.ImageCollection('projects/fisl-tundra-fire/assets/potter_fire_v20260430_ingested')

# Tidy historic data
fires_historic = fires_historic.map(lambda img: img.set('year', ee.Number.parse(img.get('year'))).rename('fire_year'))
fire_years_all = fires_historic.aggregate_array('year').distinct().sort().getInfo();
fires_historic = fires_historic.filter(ee.Filter.Or(ee.Filter.lt('year', 2001), ee.Filter.eq('year', 2025)))

# Tidy modern data
fires_modern = fires_modern.select('Burn_Mask')
fires_modern_years = fires_modern.aggregate_array('year').distinct()

def process_modern(yr):
    fires_modern_filter = fires_modern.filter(ee.Filter.eq('year', yr))
    return fires_modern_filter.max() \
                              .selfMask() \
                              .multiply(ee.Image.constant(yr)) \
                              .uint16() \
                              .rename('fire_year') \
                              .copyProperties(fires_modern_filter.first())

fires_modern = ee.ImageCollection(fires_modern_years.map(process_modern))

# Combine
fires = fires_historic.merge(fires_modern)

# Restrict to desired fire years
fire_years = ee.List.sequence(fire_start_yr, fire_end_yr).filter(ee.Filter.inList('item', fire_years_all))
fire_years_list = fire_years.getInfo()

# 1.3 LAND COVER DATA ==========

#  Macander: Before 1984, an experimental land cover products based on MSS CCDC I created
lc_1972_1984 = ee.ImageCollection('projects/foreststructure/Alaska/MSS/above_lc_mss_v20220619_tests') \
    .filter(ee.Filter.calendarRange(1972, 1984, 'year')) \
    .toBands() \
    .regexpRename('_bootstrap_20220619_mss_ccdc_1985_1986_n5000_2000_500_seed6_rf100_ml2_above_lc_co', '') \
    .regexpRename('above_lc_', 'y')

# Wang et al. land cover 1985-2014
lc_1985_2014 = ee.Image("projects/foreststructure/ABoVE/ORNL_DAAC/ABoVE_LandCover_v01")

# Macander: After 2014, using standard CCDC (probably from the same segments/coefficients used on the PFT mapping) trained on the 1985-2014 map
lc_2015_2020 = ee.ImageCollection("projects/foreststructure/ABoVE/BiomeShift/ABoVE_LC_2020") \
    .select('y2015','y2016','y2017','y2018','y2019','y2020') \
    .mosaic()

# Combine
lc_img = lc_1972_1984.addBands(lc_1985_2014).addBands(lc_2015_2020)

# Add system start and end times
lc_years_all = ee.List.sequence(1972, 2020).getInfo()
def process_lc(year):
    band = 'y' + str(year)
    next_year = year + 1
    return lc_img.select(band) \
        .set({
            'system:time_start': ee.Date.fromYMD(year, 1, 1).millis(),
            'system:time_end': ee.Date.fromYMD(next_year, 1, 1).millis(),
            'year': year
        }).rename('lc')
lc = ee.ImageCollection([process_lc(y) for y in lc_years_all])

# 1.4 PFT DATA ==========

pft = (ee.ImageCollection("projects/foreststructure/ABoVE/BiomeShift/Alaska_Yukon_PFT_202207_Filled")
         .select('cover')
         .toBands()
         .regexpRename('_20210911_pft_noRatePreds', '')
         .regexpRename('_v20220325_alaska_yukon', '')
         .regexpRename('_cover', ''))

# =============================================
# 2. EXTRACT DATA ACROSS FIRE YEARS ===========
# =============================================

# TEMP +++++++++++++++++++++++++++++++++++
fire_years_list = [1986]
# TEMP +++++++++++++++++++++++++++++++++++

for fire_year in fire_years_list:
    
    # 2.1 GET AND TIDY CURRENT YEAR FIRES ==========

    # Get current fire year fires
    fire_year_img = fires.filter(ee.Filter.eq('year', fire_year)).first()
    
    # 2.2 GET PRE-FIRE LAND COVER DATA ==========

    # Get land cover data for the year before fire
    prefire_year = fire_year - 1
    if prefire_year < 1972 or prefire_year > 2020:
        prefire_lc = ee.Image(0).rename('lc')
    else:
        prefire_lc = lc.filter(ee.Filter.calendarRange(prefire_year, prefire_year, 'year')) \
            .first().select('lc').unmask(0).multiply(10)
    
    # 2.3 CREATE FINAL DATA STACK ==========

    # Stack data
    stack = prefire_lc.addBands(pft) \
                      .updateMask(fire_year_img) \
                      .updateMask(roi_mask)
        
    # 2.4 SUMMARIZE PFT COVER ACROSS LAND COVER TYPES ==========

    # Specify PFT names and percentile labels
    pft_names = pft.bandNames()
    p_labels = ['p2', 'p3', 'p50', 'p97', 'p98']
    
    # Perform reduction
    # Group by prefire land cover
    # Calculate median and confidence intervals per PFT
    stats = stack.reduceRegion(
        reducer=ee.Reducer.percentile([2, 3, 50, 97, 98]).unweighted().repeat(pft_names.length()).group(0),
        geometry=roi.geometry(),
        scale=scale,
        maxPixels=1e12
    )
        
    # 2.5 TIDY ==========

    # Define function to turn messy dictionary into legible feature collection
    def flatten_logic(group_obj):
        
        group_obj = ee.Dictionary(group_obj)
        lc_code = group_obj.get('group')

        def map_percentiles(p_label):
            
            p_values = ee.List(group_obj.get(p_label))

            def map_pfts(pft_name):
                
                pft_index = pft_names.indexOf(pft_name)
                value = p_values.get(pft_index)
                
                return ee.Feature(None, {
                    'land_cover': lc_code,
                    'percentile': p_label,
                    'pft_type': pft_name,
                    'cover_value': value,
                    'fire_year': fire_year
                })
            
            return pft_names.map(map_pfts)
        
        return ee.List(p_labels).map(map_percentiles)

    # Apply function to tidy to legible feature collection
    groups = ee.List(stats.get('groups'))
    final_fc = ee.FeatureCollection(groups.map(flatten_logic).flatten())
    
    # 2.6 EXPORT ==========

    task = ee.batch.Export.table.toDrive(
        collection=final_fc,
        description=f'lc_pft_fire{fire_year}_{version}',
        folder='pft_trajectories',
        fileFormat='CSV'
    )

#     task.start()
    print(f"Aggregated and summaized PFT data for fire year {fire_year}. Task ready.")

Aggregated and summaized PFT data for fire year 1986. Task ready.
